# NH3 / H2 Qubit Hamiltonian (Colab Minimal)

This notebook now defers all logic to the repository code. Workflow:

1. Install pinned dependencies (Qiskit 2.x, qiskit-nature, PySCF).
2. Clone the repo.
3. Run the CLI script for NH3 (active space, 6 qubits) or H2 (4 qubits).
4. (Optional) Use fallback only (`--force-precomputed`) if PySCF fails.

See repository README for details and provenance notes.


In [ ]:
# Install pinned dependencies (single step) and import modules
import sys, subprocess, importlib
pkgs = ['qiskit==2.1.2','qiskit-nature==0.7.2','pyscf==2.6.1']
subprocess.check_call([sys.executable,'-m','pip','install','--upgrade','--no-cache-dir']+pkgs)

# Core imports (moved here as requested)
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp

print('\nVersions:')
for mod in ['qiskit','qiskit_nature','pyscf']:
    try:
        m = importlib.import_module(mod)
        print(f'  {mod}:', getattr(m,'__version__','?'))
    except Exception as e:
        print(f'  {mod}: MISSING ({e})')

Direct NH3 6-qubit active-space build and Pauli expansion (no error handling).

In [ ]:
# NH3 Pauli expansion (direct, minimal) with optional padding
import os, sys, itertools
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper

DEFAULT_MIN = 400
try:
    user_in = input(f"Minimum number of Pauli strings to print (ENTER for {DEFAULT_MIN}): ").strip()
    DESIRED_MIN = int(user_in) if user_in else DEFAULT_MIN
    if DESIRED_MIN < 1: DESIRED_MIN = DEFAULT_MIN
except Exception:
    DESIRED_MIN = DEFAULT_MIN
print(f"Target minimum terms: {DESIRED_MIN}")

# Build NH3 active-space (6 qubits)
geom = (
    'N  0.0000  0.0000  0.0000;'
    ' H  0.9377  0.0000 -0.3816;'
    ' H -0.4688  0.8119 -0.3816;'
    ' H -0.4688 -0.8119 -0.3816'
)
driver = PySCFDriver(atom=geom, basis='sto3g', charge=0, spin=0, unit=DistanceUnit.ANGSTROM)
res = driver.run()
problem = res if isinstance(res, ElectronicStructureProblem) else ElectronicStructureProblem(res)
transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
problem = transformer.transform(problem)
ferm = problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
op = mapper.map(ferm)

labels = op.paulis.to_labels()
coeffs = op.coeffs
terms = []
seen = set()
for lbl, c in zip(labels, coeffs):
    if set(lbl) == {'I'}: continue
    seen.add(lbl)
    if abs(c.imag) > 1e-12:
        terms.append((lbl, c.real, c.imag))
    else:
        terms.append((lbl, c.real, 0.0))

# Pad with zero-real labels if needed
if len(terms) < DESIRED_MIN:
    for cand in (''.join(p) for p in itertools.product('IXYZ', repeat=op.num_qubits)):
        if cand in seen or set(cand)=={'I'}: continue
        seen.add(cand)
        terms.append((cand, 0.0, 0.0))
        if len(terms) >= DESIRED_MIN: break

# Print (non-zero first, then zero padding sorted)
nonzero = [t for t in terms if abs(t[1])>1e-12 or abs(t[2])>1e-12]
zeroed = [t for t in terms if abs(t[1])<=1e-12 and abs(t[2])<=1e-12]
zeroed.sort(key=lambda x: x[0])
final = nonzero + zeroed
for lbl,r,i in final:
    if abs(i)>1e-12:
        print(f'({r:+.12f}{i:+.12f}j) * {lbl}')
    else:
        print(f'{r:+.12f} * {lbl}')
print(f'\nTotal printed: {len(final)} (physical non-zero: {len(nonzero)})')

## Note
Padding adds zero-coefficient Pauli strings to reach the requested minimum; physics unaffected.

### UCCSD Ansatz (match NH3 Hamiltonian)
Build a UCCSD circuit sized to the NH3 active-space Hamiltonian using integrated package helpers.

In [ ]:
# Build NH3 Hamiltonian (6 qubits) and matching UCCSD ansatz (compact summary)
import os, sys, subprocess, numpy as np
if not os.path.exists('GroundStateFinder'):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/Kukyos/GroundStateFinder.git'])
else:
    subprocess.check_call(['git','-C','GroundStateFinder','pull','--ff-only'])

sys.path.append('GroundStateFinder/src')
from groundstate import build_molecule_qubit_hamiltonian, uccsd_for_hamiltonian, circuit_summary

nh3_geom = 'N 0 0 0; H 0.9377 0 -0.3816; H -0.4688 0.8119 -0.3816; H -0.4688 -0.8119 -0.3816'

ham = build_molecule_qubit_hamiltonian('NH3')
ansatz, params = uccsd_for_hamiltonian(nh3_geom, ham, param_scale=0.02, seed=42)

# Derive active space summary
particles = ansatz.num_particles if isinstance(ansatz.num_particles, (tuple, list)) else (ansatz.num_particles, ansatz.num_particles)
active_e = sum(particles)
spatial = ansatz.num_spatial_orbitals

print(f"Active space: {active_e} electrons, {spatial} orbitals -> {ansatz.num_qubits} qubits")
print("Parameters:", np.array2string(params, separator=' ', max_line_width=120))
print("\nCircuit (compact, high-level):")
print(circuit_summary(ansatz, max_gates=25, decompose=False))

### VQE Skeleton Integration
Demonstrate integrating the repository's VQE skeleton (`vqeskeletal.py`) with a simple optimizer plugin.

This cell will:
1. Ensure the repo clone is present / updated.
2. Import the skeleton classes.
3. Define a minimal gradient-free optimizer (coordinate search) that fits the plugin interface.
4. Build the Hamiltonian + UCCSD ansatz via the skeleton plugins.
5. Run a mock VQE (note: expectation function is a placeholder returning 0.0 in the skeleton).

You can later replace the placeholder expectation with a real Estimator evaluation and plug in a hybrid (global→local) optimizer.

In [ ]:
# Integrate VQE skeleton with a minimal coordinate-search optimizer
import importlib, os, sys, math, numpy as np

# Ensure repo clone is present (idempotent)
if not os.path.exists('GroundStateFinder'):
    import subprocess
    subprocess.check_call(['git','clone','--depth','1','https://github.com/Kukyos/GroundStateFinder.git'])
else:
    import subprocess
    subprocess.check_call(['git','-C','GroundStateFinder','pull','--ff-only'])

# Make sure BOTH the repo root (for vqeskeletal.py) and src (for groundstate pkg) are on sys.path
if 'GroundStateFinder' not in sys.path:
    sys.path.append('GroundStateFinder')
if 'GroundStateFinder/src' not in sys.path:
    sys.path.append('GroundStateFinder/src')

# Import skeleton (module name from file)
vqe_skel = importlib.import_module('vqeskeletal')
from vqeskeletal import AnsatzPlugin, HamiltonianPlugin, ZNEDenoiserPlugin, VQE

# Define a very simple optimizer plugin (plugin interface requires optimize(fn, init_params))
class CoordinateDescentOptimizer(vqe_skel.ClassicalOptimizerPlugin):
    def __init__(self, max_iters=50, step=0.1, shrink=0.5, tol=1e-6, verbose=True):
        self.max_iters = max_iters
        self.step = step
        self.shrink = shrink
        self.tol = tol
        self.verbose = verbose

    def optimize(self, objective_function, initial_params):
        params = np.array(initial_params, dtype=float)
        best_val = objective_function(params)
        step = self.step
        if self.verbose:
            print(f"Initial value: {best_val}")
        for it in range(self.max_iters):
            improved = False
            for i in range(len(params)):
                for direction in (+1, -1):
                    trial = params.copy()
                    trial[i] += direction * step
                    val = objective_function(trial)
                    if val < best_val - self.tol:
                        best_val = val
                        params = trial
                        improved = True
                        if self.verbose:
                            print(f"Iter {it} param {i} {'+' if direction>0 else '-'} step -> {best_val}")
            if not improved:
                step *= self.shrink
                if self.verbose:
                    print(f"No improvement; shrinking step to {step}")
                if step < self.tol:
                    if self.verbose:
                        print("Converged (step below tol)")
                    break
        return params

# Instantiate plugins
ansatz_plugin = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
hamiltonian_plugin = HamiltonianPlugin()
optimizer_plugin = CoordinateDescentOptimizer(max_iters=5, step=0.2, verbose=True)
zne_plugin = ZNEDenoiserPlugin()

# Build Hamiltonian and then build ansatz explicitly before requesting parameters
ham_qubit = hamiltonian_plugin.get_hamiltonian()
ansatz_plugin.build_from_hamiltonian(ham_qubit)

info = ansatz_plugin.get_ansatz_info()
print("Ansatz info:", {k: info[k] for k in ['num_qubits','num_parameters','circuit_depth','vqe_ready']})

initial = ansatz_plugin.get_initial_parameters('random_small')
print("Initial params (first 8):", initial[:8])

vqe_instance = VQE(ansatz_plugin, hamiltonian_plugin, optimizer_plugin, zne_plugin)
opt_params, energy = vqe_instance.run(initial)
print("Returned energy (placeholder):", energy)
print("Optimized params (first 8):", (opt_params[:8] if opt_params is not None else None))

print("\nNOTE: Energy is a placeholder (0.0) until estimator integration is implemented.")

In [ ]:
# Quick energy evaluation using updated VQE skeleton (Estimator-backed if available)
from importlib import reload
import vqeskeletal as vsk
reload(vsk)
from vqeskeletal import AnsatzPlugin, HamiltonianPlugin, ZNEDenoiserPlugin, VQE

ansatz_plugin2 = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
ham_plugin2 = HamiltonianPlugin()

# Explicitly build ansatz from Hamiltonian before parameter generation
ham2 = ham_plugin2.get_hamiltonian()
ansatz_plugin2.build_from_hamiltonian(ham2)

class NoOpOpt(vsk.ClassicalOptimizerPlugin):
    def optimize(self, fn, init):
        return init
vqe2 = VQE(ansatz_plugin2, ham_plugin2, NoOpOpt(), ZNEDenoiserPlugin())
params0 = ansatz_plugin2.get_initial_parameters('zero')
val = vqe2.objective_function(params0)
print('Single energy evaluation (may be Estimator or fallback):', val)

In [ ]:
# Hybrid SPSA->COBYLA VQE run
from importlib import reload
import vqeskeletal as vsk
reload(vsk)
from vqeskeletal import AnsatzPlugin, HamiltonianPlugin, ZNEDenoiserPlugin, HybridSPSAThenCOBYLA, VQE

# Build plugins
ham_plugin = HamiltonianPlugin()
ansatz_plugin = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
zne_plugin = ZNEDenoiserPlugin()
optimizer_plugin = HybridSPSAThenCOBYLA(spsa_iters=20, switch_tol=5e-3, min_spsa=10, force_cobyla=True, verbose=True)

# Build ansatz first (explicit) then run VQE
ham_sys = ham_plugin.get_hamiltonian()
ansatz_plugin.build_from_hamiltonian(ham_sys)
info = ansatz_plugin.get_ansatz_info()
print('Ansatz info (hybrid run):', {k: info[k] for k in ['num_qubits','num_parameters','circuit_depth','vqe_ready']})

initial = ansatz_plugin.get_initial_parameters('random_small')
print('Initial (first 6):', initial[:6])

vqe = VQE(ansatz_plugin, ham_plugin, optimizer_plugin, zne_plugin, verbose=True)
params, energy = vqe.run(initial)
print('\nHybrid VQE complete. Energy:', energy)
print('Optimized params (first 6):', params[:6] if params is not None else None)


In [ ]:
# (Removed) Previously used for in-notebook hot patching of vqeskeletal.VQE measurement logic.
# Now obsolete because the repository file `vqeskeletal.py` contains the finalized implementation.
# Keeping this placeholder cell to avoid execution order surprises. Safe to delete if desired.
